# 2 — Stacking / Ensemble Heterogéneo

Combina las predicciones out-of-fold de RF, Gradient Boosting, SVM y MLP mediante un **meta-learner** (Ridge o LogisticRegression). El stacking explota que cada modelo comete errores distintos en regiones diferentes del espacio de features.

Esquema:
1. CV de 5 folds sobre train → predicciones out-of-fold de cada base-model
2. Meta-learner se entrena sobre esas predicciones
3. Evaluación sobre test

Se prueba tanto para **regresión** (temperatura exacta) como para **clasificación** (20 clases).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

In [2]:
df = pd.read_csv(CSV_PATH)

META_COLS = {
    "sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio",
}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

if FEAT_JSON.exists():
    with open(FEAT_JSON) as f: FEAT_COLS = json.load(f)
else:
    FEAT_COLS = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
print(f"Features: {len(FEAT_COLS)}")

# Etiquetas
df["label_20"] = df["temperature"].astype(int).astype(str) + "C"
classes_20 = sorted(df["label_20"].unique())

def get_split(split):
    d = df[df["split"] == split]
    return (d[FEAT_COLS].values, d["label_20"].values,
            d["temperature"].values)

X_tr, y_tr_cls, y_tr_reg = get_split("train")
X_v,  y_v_cls,  y_v_reg  = get_split("val")
X_te, y_te_cls, y_te_reg = get_split("test")

# Train+val para el stacking (más datos para el CV interno)
X_tv     = np.vstack([X_tr, X_v])
y_tv_cls = np.concatenate([y_tr_cls, y_v_cls])
y_tv_reg = np.concatenate([y_tr_reg, y_v_reg])
print(f"Train+Val: {X_tv.shape}  Test: {X_te.shape}")

Features: 58
Train+Val: (27402, 58)  Test: (4836, 58)


In [3]:
# ── Pipelines de base-models ──────────────────────────────────────────────────
def base_pipe_cls():
    return [
        ("rf", Pipeline([("imp", SimpleImputer(strategy="mean")),
                         ("m", RandomForestClassifier(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=42))])),
        ("gb", Pipeline([("imp", SimpleImputer(strategy="mean")),
                         ("m", GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42))])),
        ("svm", Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("sc", StandardScaler()),
                          ("m", SVC(kernel="rbf", C=10, probability=True, random_state=42))])),
        ("mlp", Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("sc", StandardScaler()),
                          ("m", MLPClassifier(hidden_layer_sizes=(128,64), max_iter=200, early_stopping=True, random_state=42))])),
    ]

def base_pipe_reg():
    return [
        ("rf",  Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("m", RandomForestRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=42))])),
        ("gb",  Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("m", GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42))])),
        ("svr", Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("sc", StandardScaler()),
                          ("m", SVR(kernel="rbf", C=100, epsilon=1.0, gamma="scale"))])),
        ("mlp", Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("sc", StandardScaler()),
                          ("m", MLPRegressor(hidden_layer_sizes=(128,64), max_iter=200, early_stopping=True, random_state=42))])),
    ]

print("Base models definidos")

Base models definidos


In [ ]:
# ── CLASIFICACIÓN: Stacking ───────────────────────────────────────────────────
print("Entrenando StackingClassifier (cv=5)...")
stack_cls = StackingClassifier(
    estimators=base_pipe_cls(),
    final_estimator=LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", multi_class="multinomial"),
    cv=5,
    passthrough=False,
    n_jobs=-1,
)
stack_cls.fit(X_tv, y_tv_cls)
pred_cls_stack = stack_cls.predict(X_te)
acc_stack = accuracy_score(y_te_cls, pred_cls_stack)
print(f"Stacking Clasificacion Acc = {acc_stack*100:.2f}%")

# Baseline individual: sólo RF
pipe_rf_cls = Pipeline([("imp", SimpleImputer(strategy="mean")),
                        ("m", RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42))])
pipe_rf_cls.fit(X_tv, y_tv_cls)
pred_rf_cls = pipe_rf_cls.predict(X_te)
acc_rf = accuracy_score(y_te_cls, pred_rf_cls)
print(f"RF solo            Acc = {acc_rf*100:.2f}%")

Entrenando StackingClassifier (cv=5)...


In [ ]:
# ── REGRESIÓN: Stacking ───────────────────────────────────────────────────────
print("Entrenando StackingRegressor (cv=5)...")
stack_reg = StackingRegressor(
    estimators=base_pipe_reg(),
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    passthrough=False,
    n_jobs=-1,
)
stack_reg.fit(X_tv, y_tv_reg)
pred_reg_stack = stack_reg.predict(X_te)
mae_stack = mean_absolute_error(y_te_reg, pred_reg_stack)
r2_stack  = r2_score(y_te_reg, pred_reg_stack)
print(f"Stacking Regresion  MAE={mae_stack:.3f}°C  R2={r2_stack:.4f}")

pipe_rf_reg = Pipeline([("imp", SimpleImputer(strategy="mean")),
                        ("m", RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42))])
pipe_rf_reg.fit(X_tv, y_tv_reg)
pred_rf_reg = pipe_rf_reg.predict(X_te)
mae_rf = mean_absolute_error(y_te_reg, pred_rf_reg)
r2_rf  = r2_score(y_te_reg, pred_rf_reg)
print(f"RF solo             MAE={mae_rf:.3f}°C  R2={r2_rf:.4f}")

In [ ]:
# ── Figura comparativa ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Acc clasificación
ax = axes[0]
bars = ax.bar(["RF solo", "Stacking"], [acc_rf*100, acc_stack*100], color=["#2E75B6","#C00000"], alpha=0.85, width=0.5)
for b, v in zip(bars, [acc_rf*100, acc_stack*100]):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.2f}%", ha="center", fontsize=10)
ax.set_ylabel("Accuracy [%]"); ax.set_title("Clasificación (20 clases)", fontweight="bold")
ax.set_ylim(0, 115); ax.grid(axis="y", alpha=0.3)

# MAE regresión
ax = axes[1]
bars = ax.bar(["RF solo", "Stacking"], [mae_rf, mae_stack], color=["#2E75B6","#C00000"], alpha=0.85, width=0.5)
for b, v in zip(bars, [mae_rf, mae_stack]):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("MAE [°C]"); ax.set_title("Regresión — MAE", fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# Scatter predicho vs real (stacking regresión)
ax = axes[2]
ax.scatter(y_te_reg, pred_reg_stack, alpha=0.15, s=5, c="#C00000", label="Stacking")
ax.scatter(y_te_reg, pred_rf_reg, alpha=0.1, s=5, c="#2E75B6", label="RF solo")
lim = [y_te_reg.min()-2, y_te_reg.max()+2]
ax.plot(lim, lim, "k--", linewidth=1)
ax.set_xlabel("Real [°C]"); ax.set_ylabel("Predicho [°C]")
ax.set_title("Regresión — Predicho vs Real", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Stacking vs RF solo", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("stacking_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Coeficientes del meta-learner de regresión ────────────────────────────────
coefs = pd.Series(
    stack_reg.final_estimator_.coef_,
    index=["RF", "GB", "SVR", "MLP"]
).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#CC0000" if v < 0 else "#2E75B6" for v in coefs]
ax.barh(coefs.index, coefs.values, color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Peso meta-learner (Ridge)", fontsize=10)
ax.set_title("Contribución de cada base-model al stacking (regresión)", fontsize=11, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
for i, (v, name) in enumerate(zip(coefs.values, coefs.index)):
    ax.text(v + 0.001 if v >= 0 else v - 0.001, i, f"{v:.4f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
plt.savefig("stacking_coefs_metalearner.png", dpi=150, bbox_inches="tight")
plt.show()